# Mining Dispatch — Fine-Tuning YOLO en Google Colab

## Pasos previos (hacer en tu PC ANTES de abrir Colab)

### 1. Comprimir el dataset
Abre PowerShell en la carpeta del proyecto y ejecuta:
```powershell
Compress-Archive -Path "data\processed\equipment.yolo26" -DestinationPath "equipment_yolo26.zip"
```

### 2. Subir a Google Drive
Sube estos dos archivos a la raíz de tu Google Drive:
- `equipment_yolo26.zip` (el dataset comprimido)
- `models\best.pt` (el modelo actual)

### 3. Abrir este notebook en Colab
- Ve a [colab.research.google.com](https://colab.research.google.com)
- Clic en **Archivo → Subir notebook** y selecciona este archivo `.ipynb`
- Ve a **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU**
- Ejecuta las celdas en orden de arriba a abajo

---
**Backup automático:** cada 5 épocas el modelo se guarda en Drive automáticamente.
Si se corta el internet, el entrenamiento puede reanudarse desde el último checkpoint.

In [ ]:
# ── CELDA 1: Verificar GPU ────────────────────────────────────────────────────
!nvidia-smi
import torch
print(f'\nCUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram  = props.total_memory / 1024**3
    print(f'GPU: {props.name}')
    print(f'VRAM: {vram:.1f} GB')
    if vram >= 14:    # T4
        BATCH = 16
    elif vram >= 8:
        BATCH = 8
    else:
        BATCH = 4
else:
    BATCH = 4
print(f'Batch configurado: {BATCH}')

In [ ]:
# ── CELDA 2: Instalar dependencias ────────────────────────────────────────────
!pip install ultralytics -q
import ultralytics
print('ultralytics:', ultralytics.__version__)

In [ ]:
# ── CELDA 3: Montar Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, zipfile
from pathlib import Path

# ── AJUSTA ESTOS VALORES si tu zip tiene nombre diferente ────────────────────
DRIVE_ZIP        = '/content/drive/MyDrive/equipment.yolo26.zip'  # nombre real del zip
DRIVE_MODEL      = '/content/drive/MyDrive/best.pt'
DRIVE_BACKUP_DIR = '/content/drive/MyDrive/mining_checkpoints'

# Si no encuentra el zip, busca automáticamente
if not Path(DRIVE_ZIP).exists():
    print(f'ZIP no encontrado en {DRIVE_ZIP}, buscando...')
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        for f in files:
            if f.endswith('.zip') and 'equipment' in f.lower():
                DRIVE_ZIP = os.path.join(root, f)
                print(f'Encontrado: {DRIVE_ZIP}')
                break

assert Path(DRIVE_ZIP).exists(),   f'ZIP no encontrado: {DRIVE_ZIP}'
assert Path(DRIVE_MODEL).exists(), f'best.pt no encontrado: {DRIVE_MODEL}'

Path(DRIVE_BACKUP_DIR).mkdir(parents=True, exist_ok=True)
print(f'Drive OK')
print(f'ZIP     : {DRIVE_ZIP}')
print(f'Modelo  : {DRIVE_MODEL}')
print(f'Backups : {DRIVE_BACKUP_DIR}')

In [ ]:
# ── CELDA 4: Descomprimir dataset y preparar archivos ─────────────────────────
import zipfile, shutil

DATASET_DIR = Path('/content/dataset')
MODEL_LOCAL = Path('/content/best.pt')

# Descomprimir solo si no está ya descomprimido
if not (DATASET_DIR / 'equipment.yolo26').exists():
    print('Descomprimiendo dataset...')
    with zipfile.ZipFile(DRIVE_ZIP, 'r') as z:
        z.extractall(str(DATASET_DIR))
    print('Descomprimido.')
else:
    print('Dataset ya descomprimido, usando cache.')

# Copiar modelo base
shutil.copy(DRIVE_MODEL, MODEL_LOCAL)
print(f'Modelo copiado: {MODEL_LOCAL}')

# Verificar estructura
base = DATASET_DIR / 'equipment.yolo26'
for split in ['train', 'valid', 'test']:
    imgs = len(list((base / split / 'images').glob('*')))
    lbls = len(list((base / split / 'labels').glob('*.txt')))
    print(f'  {split}: {imgs} imágenes, {lbls} labels')

print('\nEstructura OK ✓')

In [ ]:
# ── CELDA 5: Crear data.yaml con rutas absolutas ──────────────────────────────
# IMPORTANTE: usar rutas absolutas — Ultralytics duplica paths relativos

base = Path('/content/dataset/equipment.yolo26')

data_yaml = f"""train: {base / 'train' / 'images'}
val:   {base / 'valid' / 'images'}
test:  {base / 'test'  / 'images'}

nc: 3
names: ['dump_truck', 'excavator', 'wheel_loader']
"""

DATA_YAML = '/content/data.yaml'
with open(DATA_YAML, 'w') as f:
    f.write(data_yaml)

print('data.yaml creado:')
print(data_yaml)

In [ ]:
# ── CELDA 6: Sistema de backup automático ─────────────────────────────────────
import shutil
from pathlib import Path

BACKUP_EVERY = 5
RUN_DIR      = Path('/content/runs/advanced_colab')
WEIGHTS_DIR  = RUN_DIR / 'weights'
BACKUP_DIR   = Path(DRIVE_BACKUP_DIR)

def backup_a_drive(trainer):
    epoca_actual = trainer.epoch + 1
    if epoca_actual % BACKUP_EVERY != 0:
        return
    try:
        for nombre in ['last.pt', 'best.pt']:
            origen = WEIGHTS_DIR / nombre
            if origen.exists():
                shutil.copy2(origen, BACKUP_DIR / f'epoch{epoca_actual:03d}_{nombre}')
                shutil.copy2(origen, BACKUP_DIR / nombre)
        print(f'  [BACKUP] Época {epoca_actual} → Drive ✓')
    except Exception as e:
        print(f'  [BACKUP] Error: {e}')

print(f'Backup cada {BACKUP_EVERY} épocas → {BACKUP_DIR}')

In [ ]:
# ── CELDA 7: Detectar si hay checkpoint para reanudar ─────────────────────────
# Si el entrenamiento se cortó, detecta automáticamente el último checkpoint.

CHECKPOINT_DRIVE = Path(DRIVE_BACKUP_DIR) / 'last.pt'
CHECKPOINT_LOCAL = Path('/content/runs/advanced_colab/weights/last.pt')
MODO_RESUME      = False
MODELO_INICIAL   = str(MODEL_LOCAL)

if CHECKPOINT_LOCAL.exists():
    print(f'Checkpoint local encontrado: {CHECKPOINT_LOCAL}')
    MODO_RESUME = True
elif CHECKPOINT_DRIVE.exists():
    print(f'Checkpoint en Drive encontrado: {CHECKPOINT_DRIVE}')
    print('Copiando a Colab para reanudar...')
    Path('/content/runs/advanced_colab/weights').mkdir(parents=True, exist_ok=True)
    shutil.copy(CHECKPOINT_DRIVE, CHECKPOINT_LOCAL)
    MODO_RESUME = True
else:
    print('Sin checkpoint previo — entrenamiento nuevo desde best.pt')

if MODO_RESUME:
    print(f'\n→ MODO RESUME: continuando desde {CHECKPOINT_LOCAL}')
    MODELO_INICIAL = str(CHECKPOINT_LOCAL)
else:
    print(f'\n→ MODO NUEVO: iniciando desde {MODELO_INICIAL}')

In [ ]:
# ── CELDA 8: ENTRENAMIENTO ────────────────────────────────────────────────────
from ultralytics import YOLO
import torch

# Re-calcular BATCH por si la sesión se reinició
if 'BATCH' not in dir():
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 0
    BATCH = 16 if vram >= 14 else (8 if vram >= 8 else 4)
    print(f'BATCH recalculado: {BATCH}')

model = YOLO(MODELO_INICIAL)
model.add_callback('on_train_epoch_end', backup_a_drive)

print('=' * 60)
print('  FINE-TUNING AVANZADO — Mining Dispatch')
print('=' * 60)
print(f'  Modelo   : {MODELO_INICIAL}')
print(f'  Resume   : {MODO_RESUME}')
print(f'  Dataset  : {DATA_YAML}')
print(f'  Backup   : cada {BACKUP_EVERY} épocas → {DRIVE_BACKUP_DIR}')
print(f'  Batch    : {BATCH}')
print('=' * 60)

results = model.train(
    data          = DATA_YAML,
    epochs        = 30,
    imgsz         = 640,
    batch         = BATCH,
    project       = '/content/runs',
    name          = 'advanced_colab',
    exist_ok      = True,
    resume        = MODO_RESUME,
    optimizer     = 'SGD',
    lr0           = 0.00005,
    lrf           = 0.01,
    momentum      = 0.937,
    weight_decay  = 0.0005,
    warmup_epochs = 3,
    freeze        = 5,
    patience      = 10,
    degrees       = 20.0,
    perspective   = 0.002,
    shear         = 5.0,
    fliplr        = 0.5,
    flipud        = 0.15,
    scale         = 0.8,
    mosaic        = 1.0,
    copy_paste    = 0.4,
    erasing       = 0.4,
    mixup         = 0.1,
    hsv_h         = 0.015,
    hsv_s         = 0.5,
    hsv_v         = 0.4,
    amp           = True,
    save          = True,
    save_period   = 5,
    workers       = 2,
    verbose       = True,
)

print('\nEntrenamiento completado.')

In [ ]:
# ── CELDA 9: Métricas finales ─────────────────────────────────────────────────
if results is not None:
    try:
        m = results.results_dict
        print('\n[MÉTRICAS FINALES]')
        print(f"  mAP50     : {m.get('metrics/mAP50(B)',    'N/A'):.4f}")
        print(f"  mAP50-95  : {m.get('metrics/mAP50-95(B)', 'N/A'):.4f}")
        print(f"  Precision : {m.get('metrics/precision(B)', 'N/A'):.4f}")
        print(f"  Recall    : {m.get('metrics/recall(B)',    'N/A'):.4f}")
    except Exception as e:
        print(f'No se pudieron obtener métricas: {e}')

In [ ]:
# ── CELDA 10: Guardar modelo final en Drive ───────────────────────────────────
import shutil
from pathlib import Path
from datetime import datetime

WEIGHTS_DIR  = Path('/content/runs/advanced_colab/weights')
BACKUP_DIR   = Path(DRIVE_BACKUP_DIR)
timestamp    = datetime.now().strftime('%Y%m%d_%H%M')

for nombre in ['best.pt', 'last.pt']:
    origen  = WEIGHTS_DIR / nombre
    if origen.exists():
        # Copia con timestamp
        destino_ts  = BACKUP_DIR / f'{timestamp}_{nombre}'
        # Copia con nombre fijo (sobreescribe la anterior)
        destino_fix = BACKUP_DIR / nombre
        shutil.copy2(origen, destino_ts)
        shutil.copy2(origen, destino_fix)
        print(f'Guardado: {destino_ts}')

# El best.pt final también como best_advanced_colab.pt
best_final = WEIGHTS_DIR / 'best.pt'
if best_final.exists():
    destino_final = BACKUP_DIR / 'best_advanced_colab.pt'
    shutil.copy2(best_final, destino_final)
    print(f'\nModelo final: {destino_final}')

print('\n[SIGUIENTE PASO]')
print('1. Descarga best_advanced_colab.pt desde tu Drive')
print('2. Cópialo a models/best_advanced.pt en tu PC')
print('3. Cambia en .env:  MODEL_PATH=models/best_advanced.pt')
print('4. Corre main.py y compara resultados')
print('5. Si mejora, renómbralo a models/best.pt')